# 01 - Tokenization

Turn raw text into integer token IDs, first with a small word-and-punctuation tokenizer and then with GPT-2 byte-pair encoding.

In [ ]:
from pathlib import Path
import re
import tiktoken


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root')


repo_root = find_repo_root()
raw_text = (repo_root / 'data' / 'the-verdict.txt').read_text(encoding='utf-8')
print(f'{len(raw_text):,} characters')
print(raw_text[:99])

In [ ]:
TOKEN_PATTERN = r"([,.:;?_!\"()']|--|\s)"


def split_text(text):
    return [piece.strip() for piece in re.split(TOKEN_PATTERN, text) if piece.strip()]


tokens = split_text(raw_text)
vocabulary = sorted(set(tokens)) + ['<|endoftext|>', '<|unk|>']
vocab = {token: index for index, token in enumerate(vocabulary)}
print(f'{len(tokens):,} tokens, {len(vocab):,} vocabulary entries')
print(tokens[:30])

In [ ]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {index: token for token, index in vocab.items()}

    def encode(self, text):
        pieces = split_text(text)
        return [self.str_to_int.get(piece, self.str_to_int['<|unk|>']) for piece in pieces]

    def decode(self, ids):
        text = ' '.join(self.int_to_str[index] for index in ids)
        return re.sub(r"\s+([,.:;?!\"()'])", r'\1', text)


simple_tokenizer = SimpleTokenizer(vocab)
sample = 'Hello, do you like tea?'
sample_ids = simple_tokenizer.encode(sample)
print(sample_ids)
print(simple_tokenizer.decode(sample_ids))

In [ ]:
gpt2_tokenizer = tiktoken.get_encoding('gpt2')
sample = 'Hello, do you like tea? <|endoftext|> PragalvaExperimentsAsUsual'
token_ids = gpt2_tokenizer.encode(sample, allowed_special={'<|endoftext|>'})
print(token_ids)
print(gpt2_tokenizer.decode(token_ids))